Load in Labelled Windows

In [ ]:
from label_reading_tools import labelled_data_loading

width = 80
height = 120

root_folder = f"D:/CNN Formatted Data - {width} x {height}"

height += 1
width += 1

# windows not fitting the correct dimentions for a CNN are immediatly filtered in this loading process
win_raw, win_filtered, labels = labelled_data_loading(root_folder, width, height)

Label Cleaning

In [ ]:
# cleaning capitalisation issues and removing special characters
labels = [x.lower() for x in labels]
labels = [''.join(c for c in s if c.isalpha()) for s in labels]

cleaned_labels = []
# removing remaining mislabels
for l in labels:
    if l not in ["c", "p", "e", "n", "u", "f"]:
        cleaned_labels.append("u")
    else:
        cleaned_labels.append(l)

labels = cleaned_labels
del cleaned_labels

from collections import Counter
print(Counter(labels))

Counter({'u': 13465, 'f': 6227, 'p': 4843, 'c': 2859, 'e': 921, 'n': 493})


Removing Unlabelled Data

In [ ]:
import numpy as np

for i in range(len(labels) - 1, -1, -1):
    if labels[i] == "u":
        labels = np.delete(labels, i)
        win_raw = np.delete(win_raw, i)
        win_filtered = np.delete(win_filtered, i)

In [ ]:
from collections import Counter
print(Counter(labels))

Counter({'e': 1})


Normalising Windows

In [2]:
# this normalises windows (between -1, 1) individually as normalising as a group leads to inconsistent normalization
import numpy as np

win_raw_normalised = []
win_filtered_normalised = []
for w, f in zip(win_raw, win_filtered):

    minVal = w.min()
    maxVal = w.max()

    w_normalised = (((w - minVal) / (maxVal - minVal)) - 0.5) * 2
    win_raw_normalised.append(w_normalised)

    minVal = f.min()
    maxVal = f.max()

    f_normalised = (((f - minVal) / (maxVal - minVal)) - 0.5) * 2
    win_filtered_normalised.append(f_normalised)

win_raw_normalised = np.array(win_raw_normalised)
win_filtered_normalised = np.array(win_filtered_normalised)

Class Selection and One Hot Encoding

In [ ]:
# footsteps vs other
for l in labels:
    if l != "f":
        labels[labels == l] = "p"

In [ ]:
# 

In [ ]:
from keras.src.utils import to_categorical

unique = np.unique(labels)
print(unique)

for i, u in enumerate(unique):
    labels[labels == u] = i

labels_ohe = to_categorical(labels, num_classes=2)

Balancing Classes (Experiment with Binary vs Multiclass)

In [ ]:
# binary balancing
win_raw_normalised_balanced = []
win_filtered_normalised_balanced = []
labels_ohe_balanced = []
labels_balanced = []

countOther = 0
for w, wf, ohe, l in zip(win_raw_normalised, win_filtered_normalised, labels_ohe, labels):
    if l[0] != 1:
        labels_balanced.append(l)
        labels_ohe_balanced.append(ohe)
        win_raw_normalised_balanced.append(w)
        win_filtered_normalised_balanced.append(wf)
        countOther += 1

countFoot = 0
for w, wf, ohe, l in zip(win_raw_normalised, win_filtered_normalised, labels_ohe, labels):
    if l[0] == 1:
        if countFoot != countOther:
            labels_balanced.append(l)
            labels_ohe_balanced.append(ohe)
            win_raw_normalised_balanced.append(w)
            win_filtered_normalised_balanced.append(wf)
            countFoot += 1


labels_ohe_balanced = np.array(labels_ohe_balanced)
labels_balanced = np.array(labels_balanced)

win_raw_normalised_balanced = np.array(win_raw_normalised_balanced)
win_filtered_normalised_balanced = np.array(win_filtered_normalised_balanced)

In [ ]:
# multiclass balancing - to be written

Reshape Window For CNN

In [ ]:
win_raw_normalised = np.array(win_raw_normalised).astype(np.float32)
win_raw_normalised = win_raw_normalised.reshape(-1, 1, height, width)

win_filtered_normalised = np.array(win_filtered_normalised).astype(np.float32)
win_filtered_normalised = win_filtered_normalised.reshape(-1, 1, height, width)